In [4]:
import sys
print("python:", sys.version.split()[0])
print("kernel:", sys.executable, "\n")

for name in ["coffea", "uproot", "awkward", "numpy", "matplotlib", "hist", "mplhep", "vector"]:
    try:
        mod = __import__(name)
        print(f"  [ok]      {name:<12s} {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  [MISSING] {name}")

python: 3.10.14
kernel: /depot/cms/conda_envs/bhanda25/Coffea-Santosh/bin/python 

  [ok]      coffea       2024.2.2
  [ok]      uproot       5.3.10
  [ok]      awkward      2.6.5
  [ok]      numpy        1.26.4
  [ok]      matplotlib   3.9.1
  [ok]      hist         2.7.3
  [ok]      mplhep       0.3.50
  [ok]      vector       1.4.1


In [5]:
import os
import pathlib
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

# coffea 2024.2.2 warns on import that a future release will hand its four-vector
# maths to `vector`, the package you used in Lab 1 (more on that in §1.3). Nothing
# we do is affected, so we silence that one warning. Silencing warnings you have
# not read is a bad habit; this one has been read.
warnings.filterwarnings("ignore", category=FutureWarning, module="coffea.*")

import awkward as ak
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import coffea
from coffea import processor
from coffea.nanoevents import NanoEventsFactory, BaseSchema
from coffea.nanoevents.methods import candidate

from utils import chunk_jobs, plot_dimuon_spectrum

hep.style.use("CMS")
print("coffea", coffea.__version__)

coffea 2024.2.2


In [2]:
import json
import urllib.request

RECID = 75589
with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{RECID}", timeout=60) as r:
    meta = json.load(r)["metadata"]

print("title      :", meta["title"])
print("DOI        :", meta["doi"])
print("licence    :", meta["license"])
print("distribution:", json.dumps(meta["distribution"], indent=2))
meta_files = []
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        #print("Files:", meta["_file_indices"][i]['files'][j])
        meta_files.append(meta["_file_indices"][i]['files'][j])
print(meta_files[0:3])

title      : /ZZTo4L_TuneCP5_13TeV_powheg_pythia8/RunIISummer20UL16NanoAODv9-106X_mcRun2_asymptotic_v17-v1/NANOAODSIM
DOI        : 10.7483/OPENDATA.CMS.3Z2W.O02W
licence    : {'attribution': 'CC0-1.0'}
distribution: {
  "formats": [
    "nanoaodsim",
    "root"
  ],
  "number_events": 52104000,
  "number_files": 99,
  "size": 71479443458
}
[{'availability': 'online', 'bucket': 'db3870f1-c425-4e5e-83a3-6350604fa413', 'checksum': 'adler32:6af0d28f', 'file_id': '227c3166-3a84-4c93-b561-9e60463cc0c4', 'filename': '051E9D22-4F30-8E49-8477-644E65768282.root', 'key': 'CMS_mc_RunIISummer20UL16NanoAODv9_ZZTo4L_TuneCP5_13TeV_powheg_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v1_2430000_file_index.json_0', 'size': 232295116, 'tags': {}, 'uri': 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/ZZTo4L_TuneCP5_13TeV_powheg_pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v1/2430000/051E9D22-4F30-8E49-8477-644E65768282.root', 'version_id': '3296448b-1ed8-401b-884c-f0256bdc4ea7'

In [ ]:
import pathlib
import uproot

background = []
for i in range(5):
    for j in range(len(meta["_file_indices"][i]['files'])):
        # YOUR CODE HERE
        XROOTD_URL = meta_files[i]["uri"]
        HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
        EXPECTED   = meta_files[i]["size"]
        
        DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_background_data"
        # DATA_DIR.mkdir(parents=True, exist_ok=True)
        LOCAL_FILE = DATA_DIR / meta["_file_indices"][i]["files"][j]["key"]
        
        try:
            FILE = str(LOCAL_FILE)                      # from Part 2
            assert pathlib.Path(FILE).exists()
            print("Using local file from Part 2")
        except (NameError, AssertionError):
            FILE = ("root://eospublic.cern.ch//eos/opendata/cms/derived-data/"
                    "AOD2NanoAODOutreachTool/ForHiggsTo4Leptons/ZZTo4mu.root")
            # /eos/opendata/cms/dataset-semantics/NanoAODSIM/75589/ZZTo4L_TuneCP5_13TeV_powheg_pythia8_doc.root
            print("Local file not found - streaming over XRootD (slower)")
        
        background.append(uproot.open(FILE)["Events"])

count = 0
for i in range(len(background)):
    count += background[i].num_entries
#print(f"\nnumber of events: {(events[0].num_entries)}")
print("Number of Events:", count)

In [8]:
RECID = 37728
with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{RECID}", timeout=60) as r:
    meta = json.load(r)["metadata"]

print("title      :", meta["title"])
print("DOI        :", meta["doi"])
print("licence    :", meta["license"])
print("distribution:", json.dumps(meta["distribution"], indent=2))
meta_files = []
for j in range(len(meta["_file_indices"][0]['files'])):
    #print("Files:", meta["_file_indices"][i]['files'][j])
    meta_files.append(meta["_file_indices"][0]['files'][j])
print(meta_files[0:3])

title      : /GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8/RunIISummer20UL16NanoAODv9-106X_mcRun2_asymptotic_v17-v2/NANOAODSIM
DOI        : 10.7483/OPENDATA.CMS.4GPD.HQ5T
licence    : {'attribution': 'CC0-1.0'}
distribution: {
  "formats": [
    "nanoaodsim",
    "root"
  ],
  "number_events": 1000000,
  "number_files": 8,
  "size": 1696039660
}
[{'availability': 'online', 'bucket': 'b5a9ba36-e6a9-4ddb-bc0d-30f5415b0fb6', 'checksum': 'adler32:5387cdcc', 'file_id': '71753ebe-bad9-40e4-bb50-b3ed3ea9139b', 'filename': '34976A9F-8C57-044D-BB81-41704D8B017F.root', 'key': 'CMS_mc_RunIISummer20UL16NanoAODv9_GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8_NANOAODSIM_106X_mcRun2_asymptotic_v17-v2_260000_file_index.json_0', 'size': 9521067, 'tags': {}, 'uri': 'root://eospublic.cern.ch//eos/opendata/cms/mc/RunIISummer20UL16NanoAODv9/GluGluHToZZTo4L_M125_TuneCP5_13TeV_powheg2_JHUGenV7011_pythia8/NANOAODSIM/106X_mcRun2_asymptotic_v17-v2/260000/34976A9F-8C57-044D-BB8

In [10]:
signal = []
for j in range(len(meta["_file_indices"][0]['files'])):
    # YOUR CODE HERE
    XROOTD_URL = meta_files[0]["uri"]
    HTTPS_URL  = XROOTD_URL.replace("root://eospublic.cern.ch/", "https://eospublic.cern.ch/")
    EXPECTED   = meta_files[0]["size"]
    
    DATA_DIR = pathlib.Path(os.environ.get("CLUSTER_SCRATCH", pathlib.Path.home())) / "higgs_signal_data"
    # DATA_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_FILE = DATA_DIR / meta_files[j]["key"]
    
    try:
        FILE = str(LOCAL_FILE)                      # from Part 2
        assert pathlib.Path(FILE).exists()
        print("Using local file")
    except (NameError, AssertionError):
        FILE = ("root://eospublic.cern.ch//eos/opendata/cms/derived-data/"
                "AOD2NanoAODOutreachTool/ForHiggsTo4Leptons/ZZTo4mu.root")
        # /eos/opendata/cms/dataset-semantics/NanoAODSIM/75589/ZZTo4L_TuneCP5_13TeV_powheg_pythia8_doc.root
        print("Local file not found - streaming over XRootD (slower)")
    
    signal.append(uproot.open(FILE)["Events"])
count = 0
for i in range(len(signal)):
    count += signal[i].num_entries
#print(f"\nnumber of events: {(events[0].num_entries)}")
print("Number of Events:", count)


Using local file
Using local file
Using local file
Using local file
Using local file
Using local file
Using local file
Using local file
Number of Events: 1000000


In [11]:
events = [background, signal]

In [14]:
print(events)

[[<TTree 'Events' (1504 branches) at 0x7fe67714ba30>, <TTree 'Events' (1504 branches) at 0x7fe675d36170>, <TTree 'Events' (1504 branches) at 0x7fe674dc4b50>, <TTree 'Events' (1504 branches) at 0x7fe6733ef010>, <TTree 'Events' (1504 branches) at 0x7fe671998f10>, <TTree 'Events' (1504 branches) at 0x7fe67082b8b0>, <TTree 'Events' (1504 branches) at 0x7fe66f0597b0>, <TTree 'Events' (1504 branches) at 0x7fe66e093b50>, <TTree 'Events' (1504 branches) at 0x7fe66cef9f30>, <TTree 'Events' (1504 branches) at 0x7fe66bf4c8b0>, <TTree 'Events' (1504 branches) at 0x7fe66a6527d0>, <TTree 'Events' (1504 branches) at 0x7fe6696e51b0>, <TTree 'Events' (1504 branches) at 0x7fe667aa3070>, <TTree 'Events' (1504 branches) at 0x7fe666b399f0>, <TTree 'Events' (1504 branches) at 0x7fe6651cff10>, <TTree 'Events' (1504 branches) at 0x7fe6639de410>, <TTree 'Events' (1504 branches) at 0x7fe66203c310>, <TTree 'Events' (1504 branches) at 0x7fe66107e6b0>, <TTree 'Events' (1504 branches) at 0x7fe6600d4a90>, <TTree 'Ev